# 01 · Reading & Writing Files (File I/O)

**Module 11 – Files & OS** · Estimated time: 50 min

> Merges 📗 *Files* with 📘 *File I/O in Python* and adds encodings, binary files, `seek/tell`, the `x` mode, and the 🆕 Python 3.15 UTF-8 default.

### 🎯 What you will learn
- Opening, reading, writing and appending files
- All file modes: `r`, `w`, `a`, `x`, `b`, `+`
- ✅ Always use `with`
- Reading methods: `read()`, `readline()`, `readlines()`, and iterating line by line
- The file cursor: `seek()` and `tell()`
- Absolute vs relative paths
- Text encodings (UTF-8) and binary files

## 1. File I/O workflow (📘)
1. **Open** the file → creates a connection (a *file object*)
2. **Read or write**
3. **Close** it → releases the OS resource and flushes data to disk

✅ `with open(...) as f:` does step 3 for you, even if an error occurs.

## 2. Creating a practice file

📗 The bootcamp uses the IPython magic `%%writefile`, which works **only in Jupyter**:

In [1]:
%%writefile test.txt
Hello, this is a quick test file.
This is the second line of the file.

Writing test.txt


In normal Python, write files with `open(..., "w")` (section 4).

## 3. Reading

In [2]:
my_file = open("test.txt", encoding="utf-8")      # mode defaults to "r" (read, text)
print(my_file.read())
print(repr(my_file.read()))                       # '' → the cursor is at the END now
my_file.seek(0)                                   # move the cursor back to the start
print(my_file.read()[:20])
my_file.close()                                   # ❗ always close (or use with)

Hello, this is a quick test file.
This is the second line of the file.

''
Hello, this is a qui


### ✅ With `with` (recommended)

In [3]:
with open("test.txt", encoding="utf-8") as f:
    content = f.read()
print(content)
print("closed?", f.closed)

Hello, this is a quick test file.
This is the second line of the file.

closed? True


### Reading methods

| Method | Returns |
|---|---|
| `f.read()` | the whole file as one string |
| `f.read(n)` | the next `n` characters |
| `f.readline()` | the next line (including `\n`), `''` at the end |
| `f.readlines()` | a list of all lines |
| `for line in f:` | ✅ line by line, memory-efficient for huge files |

In [4]:
with open("test.txt", encoding="utf-8") as f:
    print(f.readlines())

with open("test.txt", encoding="utf-8") as f:
    print(repr(f.readline()))
    print(repr(f.read(7)))

with open("test.txt", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):      # (📗 Iterating through a File)
        print(line_number, line.rstrip("\n"))            # strip the trailing newline

['Hello, this is a quick test file.\n', 'This is the second line of the file.\n']
'Hello, this is a quick test file.\n'
'This is'
1 Hello, this is a quick test file.
2 This is the second line of the file.


With the walrus operator (📘):

In [5]:
with open("test.txt", encoding="utf-8") as f:
    while (line := f.readline()):
        print(line.strip())

Hello, this is a quick test file.
This is the second line of the file.


### Reading a file that doesn't exist

In [6]:
try:
    with open("my_file.txt", encoding="utf-8") as file:
        print(file.read())
except FileNotFoundError:
    print("File not found.")

File not found.


## 4. Writing and appending

### File modes

| Mode | Meaning | If file exists | If missing |
|---|---|---|---|
| `"r"` | read (default) | read from start | ❌ `FileNotFoundError` |
| `"w"` | write | ⚠️ **erased (truncated)** | created |
| `"a"` | append | write at the end | created |
| `"x"` | exclusive create | ❌ `FileExistsError` | created |
| `"r+"` | read **and** write | cursor at start | ❌ error |
| `"w+"` | write **and** read | ⚠️ erased | created |
| `"a+"` | append **and** read | cursor at end | created |
| add `"b"` | binary (`"rb"`, `"wb"`) | bytes instead of str | |

> ⚠️ **Use caution (📗):** opening with `"w"` or `"w+"` **truncates** the file. All existing content is gone instantly!

In [7]:
with open("new_file.txt", "w", encoding="utf-8") as f:
    f.write("Hello, world!\n")           # write() doesn't add newlines for you
    f.write("This is a new line.\n")
    count = f.write("write() returns the number of characters written\n")
print(count)

with open("new_file.txt", "a", encoding="utf-8") as f:      # append
    f.write("This is appended text.\n")
    f.writelines(["line A\n", "line B\n"])                   # write a list of strings

print(open("new_file.txt", encoding="utf-8").read())

49
Hello, world!
This is a new line.
write() returns the number of characters written
This is appended text.
line A
line B



In [8]:
with open("printed.txt", "w", encoding="utf-8") as f:
    print("Hi there", file=f)            # print() can write to files too (adds the newline)
lines = ["apple", "banana", "cherry"]
with open("fruits.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print(open("fruits.txt", encoding="utf-8").read().splitlines())

['apple', 'banana', 'cherry']


### Read **and** write: `"r+"` / `"w+"` (📗)

In [9]:
with open("test.txt", "w+", encoding="utf-8") as f:
    f.write("This is a new first line\n")
    f.seek(0)                            # go back to read what we wrote
    print(f.read())

This is a new first line



### `"x"`: safe creation (never overwrite)

In [10]:
try:
    with open("test.txt", "x", encoding="utf-8") as f:
        f.write("won't happen")
except FileExistsError:
    print("test.txt already exists, not overwritten")

test.txt already exists, not overwritten


### Appending with `%%writefile -a` (📗, Jupyter only)

In [11]:
%%writefile -a test.txt
This is text being appended to test.txt
And another line here.

Appending to test.txt


In [12]:
print(open("test.txt", encoding="utf-8").read())

This is a new first line
This is text being appended to test.txt
And another line here.



## 5. The cursor: `tell()` and `seek()`

In [13]:
with open("fruits.txt", encoding="utf-8") as f:
    print(f.tell())          # position 0
    f.read(5)
    print(f.tell())          # position after 5 characters
    f.seek(0)
    print(f.readline().strip())

0
5
apple


## 6. File paths

- **Relative path**: relative to the current working directory: `"data/report.csv"`
- **Absolute path**: from the root: `"/Users/asha/data/report.csv"` or `r"C:\Users\asha\data\report.csv"`

In [14]:
import os
print(os.getcwd())                              # where relative paths start
print(os.path.abspath("test.txt"))

/private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmpv89pgatp
/private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmpv89pgatp/test.txt


> 💡 **Windows paths:** use raw strings `r"C:\Users\name\file.txt"`, forward slashes `"C:/Users/name/file.txt"`, or (best) **`pathlib`** (next lesson), which handles every OS correctly.

## 7. Encodings 🆕

Text files are stored as **bytes**; an **encoding** maps characters ↔ bytes. UTF-8 handles every language and emoji.

In [15]:
with open("unicode.txt", "w", encoding="utf-8") as f:
    f.write("Café ☕ नमस्ते 😀\n")
with open("unicode.txt", encoding="utf-8") as f:
    print(f.read())
print(os.path.getsize("unicode.txt"), "bytes on disk")

Café ☕ नमस्ते 😀

34 bytes on disk


In [16]:
try:
    with open("unicode.txt", encoding="ascii") as f:     # wrong encoding
        f.read()
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e)

with open("unicode.txt", encoding="ascii", errors="replace") as f:
    print(f.read())

UnicodeDecodeError: 'ascii' codec can't decode byte 0xc3 in position 3: ordinal not in range(128)
Caf�� ��� ������������������ ����



> 🆕 **Python 3.15 (PEP 686): UTF-8 is now the default** for `open()` when you don't pass `encoding=`. Before 3.15, the default depended on your OS locale (e.g. `cp1252` on many Windows machines), a classic source of bugs. ✅ Keep writing `encoding="utf-8"` explicitly so your code also works on older Pythons. Use `encoding="locale"` if you really want the old behaviour.

In [17]:
import locale, sys
print("Python:", sys.version_info[:2], "| UTF-8 mode:", sys.flags.utf8_mode,
      "| preferred locale encoding:", locale.getpreferredencoding(False))

Python: (3, 15) | UTF-8 mode: 1 | preferred locale encoding: utf-8


## 8. Binary files

Images, PDFs, zip files, etc. aren't text, so open them with `"b"`. You read and write `bytes`.

In [18]:
with open("example.jpg", "rb") as f:          # an image from the course assets
    header = f.read(4)
print(header, header.hex())                  # JPEG files start with FF D8 FF

with open("example.jpg", "rb") as src, open("copy.jpg", "wb") as dst:
    while chunk := src.read(64 * 1024):      # copy in 64 KB chunks
        dst.write(chunk)
print(os.path.getsize("example.jpg") == os.path.getsize("copy.jpg"))

b'\xff\xd8\xff\xe0' ffd8ffe0
True


## 9. Practical example: word statistics of a file

In [19]:
from collections import Counter
text = """Python is great. Python is readable.
Readable code is maintainable code."""
with open("article.txt", "w", encoding="utf-8") as f:
    f.write(text)

with open("article.txt", encoding="utf-8") as f:
    words = [w.strip(".,").lower() for line in f for w in line.split()]
print("lines:", text.count("\n") + 1, "| words:", len(words))
print(Counter(words).most_common(3))

lines: 2 | words: 11
[('is', 3), ('python', 2), ('readable', 2)]


## 📝 Summary
- `open(path, mode, encoding="utf-8")`; ✅ always inside `with`.
- `r` read, `w` overwrite, `a` append, `x` create-only, `+` read & write, `b` binary.
- Iterate `for line in f` for big files; `read()` for small ones.
- `seek()`/`tell()` move/report the cursor.
- 🆕 UTF-8 is the default from 3.15, but pass `encoding` explicitly.

## 🏋️ Practice
1. Write 5 lines of your choice to `notes.txt`, then read it back and print each line numbered.
2. Append the current date to `notes.txt`.
3. Count how many lines in `notes.txt` contain the word `"python"` (case-insensitive).
4. Copy `notes.txt` to `notes_upper.txt` with all text uppercased.
5. Write a function `tail(path, n)` that returns the last `n` lines of a file.

<details><summary>Show solutions</summary>

```python
from datetime import date

with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("I love Python\nFiles are easy\npython rocks\nwith closes files\nbye\n")
with open("notes.txt", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        print(i, line.rstrip())

with open("notes.txt", "a", encoding="utf-8") as f:
    f.write(f"{date.today()}\n")

with open("notes.txt", encoding="utf-8") as f:
    print(sum("python" in line.lower() for line in f))

with open("notes.txt", encoding="utf-8") as src, open("notes_upper.txt", "w", encoding="utf-8") as dst:
    dst.write(src.read().upper())

def tail(path, n):
    with open(path, encoding="utf-8") as f:
        return f.readlines()[-n:]
```
</details>